In [7]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

In [8]:
df = pd.read_csv("processed_signature/processed/signature_dataset.csv")

In [9]:
def load_image(img_path):
    img = tf.keras.preprocessing.image.load_img(
        img_path, target_size=(224,224)
    )
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = img / 255.0
    return img

In [14]:
print(df.columns)

Index(['image_path', 'person_id', 'label', 'split'], dtype='str')


In [15]:
#Create Triplets

def create_triplets(df):

    triplets = []
    grouped = df.groupby("person_id")

    for writer_id, group in grouped:

        genuine = group[group['label'] == 0]['image_path'].values
        forged = group[group['label'] == 1]['image_path'].values

        if len(genuine) < 2:
            continue

        for i in range(len(genuine)-1):

            anchor = genuine[i]
            positive = genuine[i+1]

            # choose negative from forged OR other writer
            negative_candidates = df[df['person_id'] != writer_id]['image_path'].values
            negative = np.random.choice(negative_candidates)

            triplets.append([anchor, positive, negative])

    return triplets

In [16]:
triplets = create_triplets(df)
print("Total Triplets:", len(triplets))

Total Triplets: 1075


In [24]:
#Convert Triplets to Arrays

def generate_triplet_data(triplets):

    anchor_images = []
    positive_images = []
    negative_images = []

    for a, p, n in triplets:
        anchor_images.append(load_image(a))
        positive_images.append(load_image(p))
        negative_images.append(load_image(n))

    return (
        np.array(anchor_images),
        np.array(positive_images),
        np.array(negative_images)
    )

In [27]:
print(triplets[0])

['train/001/001_15.PNG', 'train/001/001_05.PNG', 'test/054_forg/02_0207054.PNG']


In [31]:
import os
print(os.getcwd())

c:\Users\aviru\OneDrive\Desktop\MyCodes\python\ML


In [42]:
BASE_DIR = r"c:\Users\aviru\OneDrive\Desktop\MyCodes\python\ML\processed_signature\processed"

In [ ]:
import os

BASE_DIR = r"c:\Users\aviru\OneDrive\Desktop\MyCodes\python\ML\processed_signature"

print("BASE EXISTS:", os.path.exists(BASE_DIR))
print("Folders inside BASE:")
print(os.listdir(BASE_DIR))

BASE EXISTS: True
Folders inside BASE:
['processed']
